# Análise Insumo-Produto de Acidentes de Trabalho — Espírito Santo, 2015

Este notebook percorre cada etapa do pipeline multi-agente de forma interativa,
exibindo resultados intermediários a cada passo.

In [ ]:
import sys
import os

# Ensure project root is on the path when running from notebooks/
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from agents.data_engineer import (
    extract_mip,
    ingest_smartlab,
    compatibilize_sectors,
    validate_dimensions,
)
from agents.io_specialist import (
    leontief_model,
    social_extension,
    rasmussen_hirschman,
    hypothetical_extraction,
)
from agents.data_viz import (
    build_results_dataframe,
    generate_scatter_plot,
    export_excel,
)
from agents.orchestrator import run_pipeline

print('Imports OK')

## Etapa 1 — Data Engineer
### 1.1 Extração das matrizes MIP

In [ ]:
MIP_PATH     = "../data/raw/MIP_ES_2015.xlsm"
CAT_PATH     = "../data/raw/smartlab_cat_2015.csv"
DEPARA_PATH  = "../data/raw/de_para_cnae_setores.csv"

Z, Y, X = extract_mip(MIP_PATH)

print(f"Z shape : {Z.shape}")
print(f"Y shape : {Y.shape}")
print(f"X shape : {X.shape}")
print(f"\nPrimeiros 5 valores de X:\n{X[:5]}")

### 1.2 Ingestão dos microdados CAT (SmartLab)

In [ ]:
cat_raw = ingest_smartlab(CAT_PATH)
print(f"Total de CNAEs com CAT típico ES 2015: {len(cat_raw)}")
print(f"Total de acidentes: {cat_raw['cat_count'].sum():,.0f}")
cat_raw.head(10)

### 1.3 Compatibilização CNAE → setores MIP (35)

In [ ]:
cat_35 = compatibilize_sectors(cat_raw, DEPARA_PATH)
print(f"cat_35 shape : {cat_35.shape}")
print(f"Total CATs agregados : {cat_35.sum():,.0f}")
print(f"\nDistribuição por setor:\n{cat_35}")

### Gate: validação de dimensões

In [ ]:
validate_dimensions(Z, Y, X, cat_35)
print("Validação OK — todas as dimensões são 35.")

## Etapa 2 — IO Specialist
### 2.1 Matriz de coeficientes técnicos A e inversa de Leontief L

In [ ]:
A, L = leontief_model(Z, X)

print(f"A — min={A.min():.4f}, max={A.max():.4f}, mean={A.mean():.4f}")
print(f"L — min={L.min():.4f}, max={L.max():.4f}, mean={L.mean():.4f}")

# Quick sanity: diagonal of L should all be ≥ 1
print(f"\nMínimo da diagonal de L (deve ser ≥ 1): {np.diag(L).min():.4f}")

### 2.2 Extensão social — intensidade direta a e multiplicador de pegada f

In [ ]:
a, f = social_extension(cat_35, X, L)

print("Intensidade direta a (primeiros 10 setores):")
print(a[:10])
print("\nMultiplicador de pegada f (primeiros 10 setores):")
print(f[:10])

### 2.3 Índice de Rasmussen-Hirschman U_j

In [ ]:
U_j = rasmussen_hirschman(L)

print("U_j (primeiros 10 setores):")
print(U_j[:10])
print(f"\nSetores com U_j > 1 (encadeamento acima da média): {(U_j > 1).sum()}")

### 2.4 Método de Extração Hipotética (HEM)

In [ ]:
delta_CAT = hypothetical_extraction(A, Y, a, X)

print("delta_CAT (primeiros 10 setores):")
print(delta_CAT[:10])
print(f"\nSetor com maior impacto HEM: índice {delta_CAT.argmax()} — ΔCATₖ = {delta_CAT.max():.2f}")

## Etapa 3 — Data Viz
### 3.1 DataFrame de resultados e classificação por quadrante

In [ ]:
SECTOR_NAMES = [f"Setor {i+1}" for i in range(35)]  # replace with actual names

df = build_results_dataframe(cat_35, a, U_j, f, delta_CAT, SECTOR_NAMES)

print(f"Shape do DataFrame: {df.shape}")
print(f"\nDistribuição de perfis:\n{df['Perfil'].value_counts()}")
df.head(10)

### 3.2 Gráfico de dispersão por quadrante de risco

In [ ]:
generate_scatter_plot(df)
print("Gráfico salvo em outputs/figures/quadrante_risco_es.pdf")

### 3.3 Exportação Excel

In [ ]:
export_excel(df)
print("Tabela exportada em outputs/tables/tabelas_resultados.xlsx")

## Pipeline completo

Executa todas as etapas de uma só vez via orquestrador.

In [ ]:
df_full = run_pipeline(
    mip_path="../data/raw/MIP_ES_2015.xlsm",
    cat_path="../data/raw/smartlab_cat_2015.csv",
    de_para_path="../data/raw/de_para_cnae_setores.csv",
    sector_names=SECTOR_NAMES,
)

print("\nTop 10 setores por multiplicador de pegada f:")
df_full.nlargest(10, "f")[["setor_nome", "f", "U_j", "delta_CAT", "Perfil"]]